# Retrieval Evaluation (Biomedical QA System)

## Aim

The aim of this notebook is to evaluate the **performance of the implemented semantic retrieval system**.

We measure how well the system retrieves relevant biomedical abstracts given a natural language query using **robust metrics**:
- Strict relevance criteria with **precision at k=5**
- **Reciprocal Rank** evaluation
- **Ranking confidence** analysis

In [1]:
# Load libraries and utils functions
import sys
import os
sys.path.append(os.path.abspath("../scripts"))
from utils.retrieval_utils import Retriever
from utils.retrieval_utils import Retriever
from utils.eval_utils import (
    load_test_queries,
    precision_at_k,
    reciprocal_rank,
    ranking_gap
)

2026-04-12 13:23:02.420567: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# Initi Retriever object
retriever = Retriever(
    "../data/processed/embeddings.npy",
    "../data/processed/pubmed_clean.csv"
)

## Evaluation strategy

We **manually define a small benchmark** of biomedical queries. Each query is associated with a set of expected keywords representing relevant biomedical concepts. 

**Evaluation metrics**:
1. **Precision** at k=5: a document is considered relevant only if it matches at least 2 biomedical keywords.
2. **Reciprocal Rank**: measures how early the first relevant document appears in the ranking. Higher values indicate better ranking quality.
3. **Ranking Gap**: measures confidence separation between top-1 and top-2 results. A high gap gives strong confidence in best match.

In [3]:
# Load queries
queries = load_test_queries("../evaluation/test_queries.json")

In [6]:
# Compute scores
# 1. Precision
precision_scores = []
# 2. Reciprocal Rank
rr_scores = []
# 3. Ranking Gap
gap_scores = []

# For each query
for q in queries:

    # Top 5 results for the query
    results = retriever.search(q["query"], top_k=5)

    # Scores computation
    precision = precision_at_k(results, q["keywords"], k=5, threshold=2)
    rr = reciprocal_rank(results, q["keywords"], threshold=2)
    gap = ranking_gap(results)

    print("Query:", q["query"])
    print("Precision at k=5:", round(precision, 3))
    print("Reciprocal Rank:", round(rr, 3))
    print("Ranking Gap:", round(gap, 3))
    print("-" * 50)

    precision_scores.append(precision)
    rr_scores.append(rr)
    gap_scores.append(gap)

Query: JAK2 inhibitors in cancer
Precision at k=5: 0.8
Reciprocal Rank: 0.5
Ranking Gap: 0.068
--------------------------------------------------
Query: EGFR targeted therapy
Precision at k=5: 1.0
Reciprocal Rank: 1.0
Ranking Gap: 0.087
--------------------------------------------------
Query: Alzheimer disease treatment targets
Precision at k=5: 1.0
Reciprocal Rank: 1.0
Ranking Gap: 0.013
--------------------------------------------------
Query: ER therapries in breast cancer
Precision at k=5: 1.0
Reciprocal Rank: 1.0
Ranking Gap: 0.009
--------------------------------------------------
Query: Parkison disease treatment targets
Precision at k=5: 0.6
Reciprocal Rank: 1.0
Ranking Gap: 0.004
--------------------------------------------------


In [5]:
print("FINAL EVALUATION")
print("Mean Precision (k=5):", sum(precision_scores) / len(precision_scores))
print("Mean Reciprocal Rank:", sum(rr_scores) / len(rr_scores))
print("Mean Ranking Gap:", sum(gap_scores) / len(gap_scores))

FINAL EVALUATION
Mean Precision (k=5): 0.8799999999999999
Mean Reciprocal Rank: 0.9
Mean Ranking Gap: 0.036231225728988646


## Interpretation

Summary of the 3 metrics:
- Mean Precision (at k=5): 0.88  
- Mean Reciprocal Rank (MRR): 0.90  
- Mean Ranking Gap: 0.036  

1. **Precision**: a mean of 0.88 indicates that on average **most of the top 5 retrieved documents are relevant** (at least 2 keyword matches). This suggests that the **retrieval system is effective** at identifying semantically relevant biomedical abstracts.

2. **Reciprocal Rank**: a mean of 0.90 shows that relevant documents are **typically ranked very high** in the list of results. In most cases, the first or second retrieved document is already relevant, which is a **strong signal of good ranking quality**.

3. **Ranking Gap**: a mean of 0.036 is quite small, indicating that the **similarity scores between the top-ranked documents are close**. This suggests that **multiple documents are similarly relevant** and the **model has limited confidence in distinguishing** between top candidates. This is not unexpected in biomedical literature, where many abstracts share similar terminology and context.

## Conclusion

The semantic retrieval system demonstrates **strong overall performance** on this biomedical dataset.

Key strengths:
- **High precision** in retrieving relevant documents
- **Effective ranking**, with relevant results appearing early

These results confirm that **sentence-transformer embeddings combined with cosine similarity provide a solid baseline** for biomedical semantic search.

This retrieval component is therefore **suitable to serve as the backbone of a downstream Question Answering (QA) system**.

## Limitations

Despite strong results, several limitations must be acknowledged:

1. **Keyword-based evaluation**: the **relevance is approximated using keyword matching** which does not fully capture semantic understanding. Indeed, documents may be marked relevant due to keyword overlap but not truly informative or missed due to synonymy or paraphrasing.

2. **Very small evaluation dataset**: the evaluation is based on a **very limited number of only 5 manually defined queries**. This may inflate performance metrics and not reflect real-world query diversity.

3. **Domain homogeneity**: the dataset consists of biomedical abstracts with **similar vocabulary and structure**. This makes retrieval easier compared to more heterogeneous corpora.

4. **Limited ranking discrimination**: the low ranking gap suggests that **the model struggles to clearly separate top results**. **This could impact downstream tasks such as answer generation.**

## Future Improvements

- Introduce human-labeled relevance judgments  
- Use LLM-based evaluation for semantic relevance  
- Expand dataset and query diversity  